#INIT

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Read stores from bronze

In [0]:
stores_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/bronze/csv/stores/stores.csv"

df_stores = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(stores_path)
)

display(df_stores)

#Check Stores

In [0]:
print("Bronze store count:", df_stores.count())

df_stores.printSchema()

display(
    df_stores.select(
        [count(when(col(c).isNull(), c)).alias(c)
         for c in df_stores.columns]
    )
)

#Clean Stores

In [0]:
df_stores_clean = (
    df_stores
    .dropDuplicates()
    .dropDuplicates(["store_id"])
    .filter(col("store_id").isNotNull())
    .withColumn("store_id", upper(trim(col("store_id"))))
    .withColumn("store_name", trim(col("store_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn("store_type", trim(col("store_type")))
)

# Validate Stores

In [0]:
print("Bronze count :", df_stores.count())
print("Silver count :", df_stores_clean.count())

display(df_stores_clean)

In [0]:
# check duplicate Ids
df_stores_clean.groupBy("store_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

#Write Stores as Delta

In [0]:
stores_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/stores/"

(
    df_stores_clean.write
    .format("delta")
    .mode("overwrite")
    .save(stores_silver_path)
)

#Verify Stores Delta

In [0]:
df_stores_silver = (
    spark.read
    .format("delta")
    .load(stores_silver_path)
)

display(df_stores_silver)

print("Silver store count:", df_stores_silver.count())  